# Recipe Generation: Seq2Seq LSTM + BPE (v3, corrected pipeline)

This notebook supersedes `important files/Recipe_generation _final.ipynb` (v2) and fixes these issues:

| # | Issue in v2 | Fix in v3 |
|---|---|---|
| 1 | **Train/test leakage.** The model was trained on the full dataset (including the 10% test rows) before testing | Split into train / val / test **before** any encoding; train on `train_df` only, validate on `val_df`, report on `test_df`; assert the ids are disjoint |
| 2 | `<start>` / `<end>` were plain text, so BPE split them into pieces (`start ⁇` in outputs) | Use SentencePiece's real BOS/EOS ids (1 / 2) |
| 3 | `pad_sequences` defaulted to `truncating="pre"`, so long recipes **lost their start token and first steps** | Explicit post-truncation that always keeps BOS…EOS |
| 4 | Padding counted in loss/accuracy, which inflated accuracy | `mask_zero=True` + per-token sample weights; accuracy is measured on real tokens only |
| 5 | BLEU on 10 samples, one metric, no baseline | Full test set: corpus BLEU, sentence BLEU, ROUGE-L, ingredient coverage, distinct-2, a **retrieval baseline**, and a **paired bootstrap** significance test |
| 6 | Step boundaries lost (steps joined by spaces) | Steps joined with ` . ` so generated text can be split back into steps |

**Runtime:** about 30 to 40 minutes on a Colab T4 GPU (Runtime → Change runtime type → GPU).

In [ ]:
!pip -q install sentencepiece nltk

## 1. Configuration

In [ ]:
import os, sys, ast, re, json, random, time
import numpy as np
import pandas as pd

SEED = 42
MAX_LEN_INPUT = 132      # BPE tokens for ingredients + tags
MAX_LEN_TARGET = 300     # BPE tokens for steps, including BOS/EOS
VOCAB_SIZE = 8000
EMBED_DIM = 256
LSTM_UNITS = 512
BATCH_SIZE = 64
EPOCHS = 10
SAMPLE_N = None          # e.g. 20000 for a quick smoke test; None = full dataset
N_EVAL = None            # None = full test set
ROUGE_N = 2000           # ROUGE-L is O(n^2) in pure Python, so it runs on a fixed random subset
OUT_DIR = "models"

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

## 2. Load data

In [ ]:
IN_COLAB = "google.colab" in sys.modules
CANDIDATES = ["RAW_recipes.csv", "RAW_recipes.csv/RAW_recipes.csv", "data/RAW_recipes.csv", "../data/RAW_recipes.csv"]
DATA_PATH = next((p for p in CANDIDATES if os.path.isfile(p)), "RAW_recipes.csv")
if not os.path.isfile(DATA_PATH) and os.path.exists("RAW_recipes.csv.zip"):
    import zipfile; zipfile.ZipFile("RAW_recipes.csv.zip").extractall(".")
if not os.path.isfile(DATA_PATH) and IN_COLAB:
    from google.colab import files
    files.upload()                                 # upload RAW_recipes.csv

raw = pd.read_csv(DATA_PATH)
print(raw.shape)
raw.head(3)

## 3. Preprocess

In [ ]:
df = raw.dropna(subset=["ingredients", "steps", "tags"]).copy()
for col in ["ingredients", "steps", "tags"]:
    df[col] = df[col].apply(ast.literal_eval)
df = df[(df["ingredients"].str.len() > 0) & (df["steps"].str.len() > 0) & (df["tags"].str.len() > 0)]

df["ingredients_text"] = df["ingredients"].apply(lambda x: " ".join(i.lower() for i in x))
df["tags_text"] = df["tags"].apply(lambda x: " ".join(t.lower().replace("-", " ") for t in x))
df["input_text"] = df["ingredients_text"] + " " + df["tags_text"]
# Keep step boundaries with " . " (fix #6). No text <start>/<end> markers (fix #2).
df["steps_text"] = df["steps"].apply(lambda x: " . ".join(s.strip().lower() for s in x if s.strip()))
df = df[df["steps_text"].str.len() > 0].reset_index(drop=True)

if SAMPLE_N:
    df = df.sample(SAMPLE_N, random_state=SEED).reset_index(drop=True)

df = df[["id", "name", "ingredients", "input_text", "steps_text"]]
print(f"{len(df):,} recipes after cleaning")
df.head(3)

## 4. Split before anything else (fix #1)

In [ ]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(df, test_size=0.10, random_state=SEED)
train_df, val_df = train_test_split(train_val_df, test_size=0.10, random_state=SEED)

assert set(train_df.id).isdisjoint(test_df.id)
assert set(train_df.id).isdisjoint(val_df.id)
assert set(val_df.id).isdisjoint(test_df.id)

dup = test_df.input_text.isin(set(train_df.input_text)).mean()
print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}")
print(f"Test inputs that appear verbatim in train (near-duplicate recipes): {dup:.2%}")

## 5. BPE tokenizer, trained on the training split only

In [ ]:
import sentencepiece as spm

corpus_path = f"{OUT_DIR}/bpe_corpus_train.txt"
with open(corpus_path, "w", encoding="utf-8") as f:
    for t in pd.concat([train_df.input_text, train_df.steps_text]):
        f.write(t + "\n")

spm.SentencePieceTrainer.Train(
    input=corpus_path, model_prefix=f"{OUT_DIR}/bpe_v3", vocab_size=VOCAB_SIZE, model_type="bpe",
    pad_id=0, bos_id=1, eos_id=2, unk_id=3,
    input_sentence_size=1_000_000, shuffle_input_sentence=True, minloglevel=2,
)
sp = spm.SentencePieceProcessor(model_file=f"{OUT_DIR}/bpe_v3.model")
PAD, BOS, EOS, UNK = sp.pad_id(), sp.bos_id(), sp.eos_id(), sp.unk_id()
assert (PAD, BOS, EOS) == (0, 1, 2)
print("pieces:", sp.get_piece_size(), "| BOS/EOS:", sp.id_to_piece(BOS), sp.id_to_piece(EOS))
print(sp.encode("add garlic and onion to the pan", out_type=str))

## 6. Encode (fixes #2 and #3)

In [ ]:
def pad_post(seqs, maxlen):
    out = np.zeros((len(seqs), maxlen), dtype=np.int32)
    for i, s in enumerate(seqs):
        s = s[:maxlen]
        out[i, :len(s)] = s
    return out

def encode_inputs(texts):
    return pad_post(sp.encode(list(texts)), MAX_LEN_INPUT)

def encode_targets(texts):
    seqs = sp.encode(list(texts))
    # Truncate the body, never BOS/EOS
    return pad_post([[BOS] + s[:MAX_LEN_TARGET - 2] + [EOS] for s in seqs], MAX_LEN_TARGET)

def make_arrays(frame):
    X = encode_inputs(frame.input_text)
    Y = encode_targets(frame.steps_text)
    y_in, y_out = Y[:, :-1], Y[:, 1:]
    sw = (y_out != PAD).astype(np.float32)          # fix #4: ignore padding
    return X, y_in, y_out, sw

Xtr, Ytr_in, Ytr_out, SWtr = make_arrays(train_df)
Xva, Yva_in, Yva_out, SWva = make_arrays(val_df)
Xte = encode_inputs(test_df.input_text)

tgt_lens = np.array([len(s) + 2 for s in sp.encode(list(train_df.steps_text))])
inp_lens = np.array([len(s) for s in sp.encode(list(train_df.input_text))])
print(f"targets truncated: {(tgt_lens > MAX_LEN_TARGET).mean():.1%} | inputs truncated: {(inp_lens > MAX_LEN_INPUT).mean():.1%}")
print(f"median target length: {np.median(tgt_lens):.0f} tokens")
assert (Ytr_in[:, 0] == BOS).all() and ((Ytr_out == EOS).sum(axis=1) == 1).all()

## 7. Model

In [ ]:
import tensorflow as tf
import keras
from keras import layers
tf.random.set_seed(SEED)
print(tf.__version__, tf.config.list_physical_devices("GPU"))

V = sp.get_piece_size()
enc_in = keras.Input((None,), dtype="int32", name="encoder_inputs")
dec_in = keras.Input((None,), dtype="int32", name="decoder_inputs")

enc_emb = layers.Embedding(V, EMBED_DIM, mask_zero=True, name="enc_emb")
enc_lstm = layers.LSTM(LSTM_UNITS, return_state=True, name="enc_lstm")
_, h, c = enc_lstm(enc_emb(enc_in))

dec_emb = layers.Embedding(V, EMBED_DIM, mask_zero=True, name="dec_emb")
dec_lstm = layers.LSTM(LSTM_UNITS, return_sequences=True, return_state=True, name="dec_lstm")
dec_seq, _, _ = dec_lstm(dec_emb(dec_in), initial_state=[h, c])
dec_dense = layers.Dense(V, activation="softmax", name="dec_dense")
probs = dec_dense(dec_seq)

model = keras.Model([enc_in, dec_in], probs, name="recipe_seq2seq_v3")
model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="sparse_categorical_crossentropy",
              weighted_metrics=["accuracy"])
model.summary()

## 8. Train (train split only; early stopping on val loss)

In [ ]:
def make_ds(X, y_in, y_out, sw, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(({"encoder_inputs": X, "decoder_inputs": y_in}, y_out, sw))
    if shuffle:
        ds = ds.shuffle(20_000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

MODEL_PATH = f"{OUT_DIR}/recipe_seq2seq_bpe_v3.keras"
callbacks = [
    keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
]
t0 = time.time()
history = model.fit(make_ds(Xtr, Ytr_in, Ytr_out, SWtr, True),
                    validation_data=make_ds(Xva, Yva_in, Yva_out, SWva, False),
                    epochs=EPOCHS, callbacks=callbacks)
print(f"training time: {(time.time() - t0) / 60:.1f} min")
json.dump(history.history, open(f"{OUT_DIR}/history_v3.json", "w"))

In [ ]:
import matplotlib.pyplot as plt
hist = history.history
ACC = next(k for k in hist if "acc" in k and not k.startswith("val_"))   # "accuracy" in Keras 3
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist["loss"], label="train"); ax[0].plot(hist["val_loss"], label="val"); ax[0].set_title("Loss (non-pad tokens)")
ax[1].plot(hist[ACC], label="train"); ax[1].plot(hist["val_" + ACC], label="val"); ax[1].set_title("Token accuracy (non-pad tokens)")
for a in ax: a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_curves_v3.png", dpi=120); plt.show()

## 9. Inference: batched greedy decoding

In [ ]:
# Works after training in this session or after keras.models.load_model(MODEL_PATH)
model = keras.models.load_model(MODEL_PATH)
enc_emb, enc_lstm = model.get_layer("enc_emb"), model.get_layer("enc_lstm")
dec_emb, dec_lstm, dec_dense = model.get_layer("dec_emb"), model.get_layer("dec_lstm"), model.get_layer("dec_dense")

@tf.function(reduce_retracing=True)
def encode_batch(x):
    _, h, c = enc_lstm(enc_emb(x), mask=tf.not_equal(x, 0))
    return h, c

@tf.function(reduce_retracing=True)
def decode_step(tok, h, c):
    out, h, c = dec_lstm(dec_emb(tok[:, None]), initial_state=[h, c])
    return tf.argmax(dec_dense(out[:, -1, :]), axis=-1, output_type=tf.int32), h, c

def greedy_decode(X, batch_size=256, max_len=MAX_LEN_TARGET - 1):
    texts = []
    for i in range(0, len(X), batch_size):
        xb = tf.constant(X[i:i + batch_size], dtype=tf.int32)
        h, c = encode_batch(xb)
        tok = tf.fill([tf.shape(xb)[0]], BOS)
        steps, done = [], np.zeros(xb.shape[0], dtype=bool)
        for t in range(max_len):
            tok, h, c = decode_step(tok, h, c)
            steps.append(tok.numpy())
            done |= steps[-1] == EOS
            if done.all():
                break
        for row in np.stack(steps, axis=1).tolist():
            texts.append(sp.decode(row[:row.index(EOS)] if EOS in row else row))
    return texts

def generate(input_text):
    return greedy_decode(encode_inputs([input_text.lower()]))[0]

print(generate("chicken breast garlic lemon olive oil salt pepper 30 minutes or less main dish"))

## 10. Evaluation set and retrieval baseline

A **non-neural baseline** puts the LSTM numbers in context. For each test recipe it finds the most similar *training* input (TF-IDF cosine similarity) and returns that recipe's steps. If the seq2seq model can't beat copying the closest existing recipe, it isn't adding much.

In [ ]:
eval_df = test_df if N_EVAL is None else test_df.sample(N_EVAL, random_state=SEED)
eval_df = eval_df.reset_index(drop=True)
refs = eval_df.steps_text.tolist()
systems = {}                       # name -> list of predicted step strings
print(f"evaluating on {len(eval_df):,} held-out recipes")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = time.time()
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr_tfidf = vec.fit_transform(train_df.input_text)
Xev_tfidf = vec.transform(eval_df.input_text)
nn_idx = np.concatenate([
    np.asarray((Xev_tfidf[i:i + 500] @ Xtr_tfidf.T).argmax(axis=1)).ravel()
    for i in range(0, Xev_tfidf.shape[0], 500)
])
systems["Retrieval baseline (TF-IDF 1-NN)"] = train_df.steps_text.iloc[nn_idx].tolist()
print(f"retrieval done in {time.time() - t0:.0f}s")

In [ ]:
t0 = time.time()
systems["Seq2seq LSTM + BPE (v3)"] = greedy_decode(encode_inputs(eval_df.input_text))
print(f"decoded {len(eval_df):,} recipes in {(time.time() - t0) / 60:.1f} min")

## 11. Metrics

- **Corpus BLEU-4**: n-gram overlap with the reference recipe, pooled over the test set.
- **Mean sentence BLEU**: the same per recipe, averaged (smoothing method 4, so it's comparable to v2's 0.040).
- **ROUGE-L F1**: longest-common-subsequence overlap, which captures step ordering.
- **Ingredient coverage**: share of the input ingredients (head noun) mentioned in the generated steps, i.e. whether the model uses what you gave it. The human-written reference gives a ceiling.
- **Distinct-2**: unique bigrams / total bigrams across outputs. Low values mean repetitive, generic text.

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
smooth = SmoothingFunction()

def words(s):
    return re.findall(r"[a-z0-9]+|[^\sa-z0-9]", s.lower())

def lcs_len(a, b):
    if len(a) < len(b): a, b = b, a
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]

def rouge_l(ref, hyp):
    if not ref or not hyp: return 0.0
    l = lcs_len(ref, hyp)
    if l == 0: return 0.0
    p, r = l / len(hyp), l / len(ref)
    return 2 * p * r / (p + r)

def _norm(w): return w[:-1] if w.endswith("s") and len(w) > 3 else w

def ingredient_coverage(ings, text):
    vocab = {_norm(w) for w in re.findall(r"[a-z]+", text.lower())}
    heads = [_norm(re.findall(r"[a-z]+", i.lower())[-1]) for i in ings if re.findall(r"[a-z]+", i.lower())]
    return float(np.mean([h in vocab for h in heads])) if heads else np.nan

def distinct_2(texts):
    grams = [tuple(w[i:i + 2]) for w in map(words, texts) for i in range(len(w) - 1)]
    return len(set(grams)) / max(len(grams), 1)

rouge_idx = np.random.RandomState(SEED).choice(len(refs), min(ROUGE_N, len(refs)), replace=False)
ref_tok = [words(r) for r in refs]

def evaluate(preds):
    hyp_tok = [words(p) for p in preds]
    sent = np.array([sentence_bleu([r], h, smoothing_function=smooth.method4) if h else 0.0
                     for r, h in zip(ref_tok, hyp_tok)])
    rl = np.array([rouge_l(ref_tok[i], hyp_tok[i]) for i in rouge_idx])
    cov = np.array([ingredient_coverage(i, p) for i, p in zip(eval_df.ingredients, preds)])
    return {
        "corpus_bleu4": corpus_bleu([[r] for r in ref_tok], hyp_tok, smoothing_function=smooth.method1),
        "sentence_bleu": sent.mean(), "rouge_l": rl.mean(),
        "ingredient_coverage": np.nanmean(cov), "distinct_2": distinct_2(preds),
        "avg_len_words": np.mean([len(h) for h in hyp_tok]),
        "_sent_bleu": sent, "_rouge_l": rl, "_coverage": cov,
    }

results = {name: evaluate(p) for name, p in systems.items()}
results["Human reference (ceiling)"] = {
    "ingredient_coverage": np.nanmean([ingredient_coverage(i, r) for i, r in zip(eval_df.ingredients, refs)]),
    "distinct_2": distinct_2(refs), "avg_len_words": np.mean([len(r) for r in ref_tok]),
}
summary = pd.DataFrame({k: {m: v for m, v in d.items() if not m.startswith("_")} for k, d in results.items()}).T
summary.round(4)

## 12. Is the difference significant? (paired bootstrap)

In [ ]:
def paired_bootstrap(a, b, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    d = a - b
    mask = ~np.isnan(d); d = d[mask]
    boots = np.array([rng.choice(d, len(d)).mean() for _ in range(n)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return d.mean(), lo, hi, max(p, 1 / n)

tests = {}
M, B = "Seq2seq LSTM + BPE (v3)", "Retrieval baseline (TF-IDF 1-NN)"
if M in results and B in results:
    for metric in ["_sent_bleu", "_rouge_l", "_coverage"]:
        diff, lo, hi, p = paired_bootstrap(results[M][metric], results[B][metric])
        tests[metric[1:]] = dict(diff=diff, ci_low=lo, ci_high=hi, p=p)
        print(f"{metric[1:]:>12}: LSTM - retrieval = {diff:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  p~{p:.4f}")

## 13. Qualitative examples

In [ ]:
for i in range(3):
    print(f"\n=== {eval_df.name[i]} ===")
    print("INPUT     :", eval_df.input_text[i][:160], "...")
    print("REFERENCE :", refs[i][:300], "...")
    for name, preds in systems.items():
        print(f"{name[:10].upper():<10}:", preds[i][:300], "...")

## 14. Save results and print a table for the README

In [ ]:
report = {
    "n_eval": len(eval_df), "n_train": len(train_df), "n_val": len(val_df), "rouge_subset": len(rouge_idx),
    "metrics": summary.where(pd.notna(summary), None).to_dict(orient="index"),
    "bootstrap_vs_retrieval": tests,
}
if "history" in globals():
    report["best_epoch"] = int(np.argmin(history.history["val_loss"]) + 1)
    report["best_val_loss"] = float(np.min(history.history["val_loss"]))
    report["best_val_accuracy"] = float(history.history["val_" + next(k for k in history.history if "acc" in k and not k.startswith("val_"))][report["best_epoch"] - 1])
json.dump(report, open(f"{OUT_DIR}/eval_metrics_v3.json", "w"), indent=2, default=float)
pd.DataFrame({"input": eval_df.input_text, "reference": refs, **systems}).head(200) \
  .to_csv(f"{OUT_DIR}/sample_predictions_v3.csv", index=False)

cols = ["corpus_bleu4", "sentence_bleu", "rouge_l", "ingredient_coverage", "distinct_2"]
print("| System | " + " | ".join(cols) + " |\n|---|" + "---|" * len(cols))
for name, row in summary[cols].iterrows():
    print(f"| {name} | " + " | ".join("–" if pd.isna(v) else f"{v:.3f}" for v in row) + " |")
print(json.dumps({k: v for k, v in report.items() if k != "metrics"}, indent=2, default=float))